Este notebook é responsável pela construção da camada **GOLD** da base Bolsa Trabalho.

Objetivos da camada GOLD:
- Consolidar os dados tratados da SILVER
- Aplicar regras de negócio
- Criar colunas analíticas
- Entregar uma tabela pronta para consumo direto no Power BI

Granularidade:
- 1 linha = 1 inscrição (OS)

In [5]:
from pyspark.sql.functions import (
    col, trim, regexp_replace, when, lit
)
from pyspark.sql.types import DoubleType

# Leitura da tabela SILVER
df_gold_bolsa_trabalho = spark.table("silver_bolsa_trabalho")

StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 7, Finished, Available, Finished)

In [6]:
# df_gold_bolsa_trabalho = df_gold_bolsa_trabalho.toPandas()

StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 8, Finished, Available, Finished)

In [7]:
# df_gold_bolsa_trabalho.filter(like="renda")

StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 9, Finished, Available, Finished)

## Padronização de colunas

Nesta etapa, renomeamos colunas para nomes curtos, claros e padronizados,
facilitando análises, SQL e integração com Power BI.


In [8]:
df_gold_bolsa_trabalho = (
    df_gold_bolsa_trabalho
    .withColumnRenamed("no_solicitacao", "os")
    .withColumnRenamed("nome_do_servico_digital", "servico")
    .withColumnRenamed("data_de_solicitacao", "data_solicitacao")
    .withColumnRenamed("data_finalizacao", "data_finalizacao")
    .withColumnRenamed("solicitante", "solicitante")
    .withColumnRenamed("status", "status")
)

df_gold_bolsa_trabalho.columns


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 10, Finished, Available, Finished)

['os',
 'data_solicitacao',
 'servico',
 'nome_do_interessado',
 'etapa_atual',
 'data_finalizacao',
 'solicitante',
 'tempo_residencia_em_osasco',
 'data_da_ultima_parcela',
 'nome_da_medida_socioeducativa_e_data_de_inicio',
 'quais_equipamentos_culturais_voce_conhece_eou_frequenta_na_regiao_que_voce_reside',
 'voce_possui_um_celular_com_acesso_a_internet',
 'cpf_do_interessado',
 'rg_do_interessado',
 'uf_rg_do_interessado',
 'idade_do_interessado',
 'telefone_do_interessado',
 'telefone_alternativo_do_interessado',
 'email_do_interessado',
 'opcao_de_curso',
 'operacao',
 'status_do_interessado',
 'status',
 'cadunico_codigo_familiar',
 'cep',
 'tipo_logradouro',
 'nome_do_logradouro',
 'numero',
 'complemento',
 'cidade',
 'bairro',
 'uf',
 'ano',
 'etapa',
 'turma',
 'atividade',
 'nome_da_mae',
 'sexo_do_interessado',
 'estado_civil_do_interessado',
 'escolaridade_do_interessado',
 'situacao_servico_militar',
 'identidade_genero',
 'raca',
 'natureza_da_moradia',
 'n_de_pessoas_n

## Construção do DataFrame GOLD Base

Seleção das colunas essenciais para análise:
- Identificação da inscrição
- Tempo e status
- Localização
- Perfil socioeconômico
- Informações do programa
- Renda familiar


In [9]:
df_gold_base = (
    df_gold_bolsa_trabalho
    .select(
        # Identificação / tempo
        col("os"),
        col("data_solicitacao"),
        col("data_finalizacao"),
        col("ano"),
        col("status"),
        col("status_do_interessado"),
        col("etapa_atual"),
        col("servico"),
        col("operacao"),

        # Localização
        col("cidade"),
        col("bairro"),
        col("uf"),

        # Perfil
        col("nome_do_interessado"),
        col("sexo_do_interessado"),
        col("idade_do_interessado"),
        col("escolaridade_do_interessado"),
        col("condicao_de_trabalho"),
        col("deficiencia"),
        col("identidade_genero"),
        col("a_familia_vive_em_condicoes_precarias_de_moradia"),
        col("natureza_da_moradia"),
        col("total_de_membros_na_familia"),
        col("possui_na_familia_pessoas_menores_de_18_anos_de_idade"),

        # Programa
        col("opcao_de_curso"),
        col("turma"),
        col("atividade"),

        # Renda
        col("renda_total_da_familia")
    )
)


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 11, Finished, Available, Finished)

## Tratamento da Renda

- Convertimos a renda para valor numérico
- Criamos a faixa de renda seguindo exatamente a regra utilizada no Power BI



In [10]:
from pyspark.sql.functions import col, trim, regexp_replace, when, floor
from pyspark.sql.types import IntegerType, DoubleType

# 1. Limpeza básica
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    trim(col("renda_total_da_familia"))
)

# 2. Padroniza decimal: vírgula vira ponto
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    regexp_replace(col("renda_total_da_familia"), ",", ".")
)

# 3. Remove separador de milhar (ponto entre números)
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    regexp_replace(col("renda_total_da_familia"), r"(?<=\d)\.(?=\d{3})", "")
)

# 4. Converte para double
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    col("renda_total_da_familia").cast(DoubleType())
)

# 5. Remove casas decimais
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    floor(col("renda_total_da_familia"))
)

# 6. Normaliza nulos, zero e negativos
df_gold_base = df_gold_base.withColumn(
    "renda_total_da_familia",
    when(
        col("renda_total_da_familia").isNull() | (col("renda_total_da_familia") <= 0),
        0
    ).otherwise(col("renda_total_da_familia"))
    .cast(IntegerType())
)

# 7. Faixa de renda baseada em inteiro
df_gold_base = df_gold_base.withColumn(
    "faixa_renda",
    when(col("renda_total_da_familia") == 0, "Sem informação")
    .when(col("renda_total_da_familia") <= 500, "Até R$ 500")
    .when(col("renda_total_da_familia") <= 1000, "R$ 501 a R$ 1.000")
    .when(col("renda_total_da_familia") <= 2000, "R$ 1.001 a R$ 2.000")
    .when(col("renda_total_da_familia") <= 3000, "R$ 2.001 a R$ 3.000")
    .when(col("renda_total_da_familia") <= 4000, "R$ 3.001 a R$ 4.000")
    .when(col("renda_total_da_familia") <= 5000, "R$ 4.001 a R$ 5.000")
    .otherwise("Acima de R$ 5.000")
)


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 12, Finished, Available, Finished)

In [11]:
df_gold_base.select("renda_total_da_familia", "faixa_renda") \
    .orderBy(col("renda_total_da_familia").desc()) \
    .show(60, False)


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 13, Finished, Available, Finished)

+----------------------+-------------------+
|renda_total_da_familia|faixa_renda        |
+----------------------+-------------------+
|11900                 |Acima de R$ 5.000  |
|6632                  |Acima de R$ 5.000  |
|6120                  |Acima de R$ 5.000  |
|5280                  |Acima de R$ 5.000  |
|5183                  |Acima de R$ 5.000  |
|4200                  |R$ 4.001 a R$ 5.000|
|3518                  |R$ 3.001 a R$ 4.000|
|3100                  |R$ 3.001 a R$ 4.000|
|3095                  |R$ 3.001 a R$ 4.000|
|3036                  |R$ 3.001 a R$ 4.000|
|3000                  |R$ 2.001 a R$ 3.000|
|2700                  |R$ 2.001 a R$ 3.000|
|2600                  |R$ 2.001 a R$ 3.000|
|2442                  |R$ 2.001 a R$ 3.000|
|1900                  |R$ 1.001 a R$ 2.000|
|1900                  |R$ 1.001 a R$ 2.000|
|1650                  |R$ 1.001 a R$ 2.000|
|1600                  |R$ 1.001 a R$ 2.000|
|1590                  |R$ 1.001 a R$ 2.000|
|1518     

In [12]:
df_gold_base.select("renda_total_da_familia").describe().show()


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 14, Finished, Available, Finished)

+-------+----------------------+
|summary|renda_total_da_familia|
+-------+----------------------+
|  count|                    70|
|   mean|    1305.8285714285714|
| stddev|     2046.779230049774|
|    min|                     0|
|    max|                 11900|
+-------+----------------------+



## Validação do DataFrame GOLD

Conferência do schema e amostra dos dados para garantir:
- Tipos corretos
- Regras aplicadas
- Compatibilidade com o Power BI


In [14]:
df_gold_base.select("renda_total_da_familia").describe().show()

StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 16, Finished, Available, Finished)

+-------+----------------------+
|summary|renda_total_da_familia|
+-------+----------------------+
|  count|                    70|
|   mean|    1305.8285714285714|
| stddev|     2046.779230049774|
|    min|                     0|
|    max|                 11900|
+-------+----------------------+



## DataFrame GOLD final – Bolsa Trabalho

DataFrame consolidado e pronto para gravação no Lakehouse.


## Escrita da Tabela GOLD

A tabela final será gravada no formato Delta


In [15]:
(
    df_gold_bolsa_trabalho
    .write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_bolsa_trabalho")
)


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 17, Finished, Available, Finished)

In [18]:
%%sql
SELECT
    renda_total_da_familia,
    COUNT(*) AS qtd
FROM gold_bolsa_trabalho
GROUP BY renda_total_da_familia
ORDER BY renda_total_da_familia DESC
LIMIT 30


StatementMeta(, f6259ed6-462c-40bc-879d-58269735fdcb, 20, Finished, Available, Finished)

<Spark SQL result set with 25 rows and 2 fields>